# 01 · Data pipeline — 10 years of surface inputs + GLORYS target → Zarr on Drive
Runs entirely on Colab. Nothing touches the laptop disk. Every step is **resumable**: re-run a cell after a
disconnect and it continues from `<store>.done.json`.

| Store | Contents | Size (approx) |
|---|---|---|
| `inputs.zarr` | SST (OSTIA), SSS (CMEMS multi-obs), SLA (DUACS), currents (OSCAR), winds (CCMP), 2014–2023 daily, 0.25° | ~1.5 GB |
| `target.zarr` | GLORYS12 θ at 15 standard depths, int16 (0.001 °C) | ~2.5 GB |
| `hycom.zarr` | HYCOM via **your GEE** (test year, independent comparator) | ~0.3 GB |
| `argo_2023.parquet` | Argo profiles for independent validation | small |

**Secrets** (🔑 icon in the left bar): `CMEMS_USER`, `CMEMS_PASS` (free at marine.copernicus.eu),
`EARTHDATA_USER`, `EARTHDATA_PASS` (free at urs.earthdata.nasa.gov). GEE uses your own login, project `syltihe`.

In [ ]:
# ── Setup: Drive (data) + GitHub (code) + deps ──────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/OceanEmbed'          # data, checkpoints, outputs live here
REPO = '/content/Sylithe_Ocean_Embedding'
import os, sys; os.makedirs(ROOT, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/karan27-dev/Sylithe_Ocean_Embedding {REPO}
sys.path.insert(0, REPO)
# Colab already ships xarray, dask, netCDF4 and earthengine-api. Install only what is missing, one
# package at a time, so pip never has to search a large version space (that is what hangs).
!pip install -q --upgrade-strategy only-if-needed copernicusmarine
!pip install -q --upgrade-strategy only-if-needed earthaccess
!pip install -q --upgrade-strategy only-if-needed argopy zarr
import importlib
for m in ['xarray', 'zarr', 'dask', 'netCDF4', 'ee', 'copernicusmarine', 'earthaccess', 'argopy']:
    print(f'{m:17s}', getattr(importlib.import_module(m), '__version__', 'ok'))

In [ ]:
# ── Credentials ───────────────────────────────────────────────────────────
# Do NOT type usernames/passwords here. Put them in the 🔑 Secrets panel (left bar) under exactly
# these four NAMES, with "Notebook access" switched on. This cell only reads them by name.
try:
    import copernicusmarine, earthaccess, ee
except ModuleNotFoundError:
    raise SystemExit('Run the Setup cell above first (it installs copernicusmarine, earthaccess, ...).')
from google.colab.userdata import SecretNotFoundError, NotebookAccessError
secrets = {}
for name in ['CMEMS_USER', 'CMEMS_PASS', 'EARTHDATA_USER', 'EARTHDATA_PASS']:
    try:
        secrets[name] = userdata.get(name)
    except (SecretNotFoundError, NotebookAccessError):
        raise SystemExit(f'Secret "{name}" is missing or has Notebook access switched off: add it in 🔑 Secrets.')
copernicusmarine.login(username=secrets['CMEMS_USER'], password=secrets['CMEMS_PASS'], force_overwrite=True)
os.environ['EARTHDATA_USERNAME'] = secrets['EARTHDATA_USER']
os.environ['EARTHDATA_PASSWORD'] = secrets['EARTHDATA_PASS']
earthaccess.login(strategy='environment')
ee.Authenticate(); ee.Initialize(project='syltihe')
print('all providers authenticated')

In [ ]:
# ── Verify dataset IDs before a multi-hour pull (IDs change with product versions) ──
from oceanembed import config as C
ids = {s.dataset_id for v in C.SOURCES.values() for s in v if s.provider == 'cmems'} | set(C.GLORYS_IDS)
for i in sorted(ids):
    try:
        ds = copernicusmarine.open_dataset(dataset_id=i, **C.BBOX)
        print(f'OK  {i}: {str(ds.time.values[0])[:10]} → {str(ds.time.values[-1])[:10]}  vars={list(ds.data_vars)[:5]}')
    except Exception as e:
        print(f'ERR {i}: {str(e)[:200]}  → fix the ID in oceanembed/config.py')
for sn in ['OSCAR_L4_OC_FINAL_V2.0', 'CCMP_WINDS_10M6HR_L4_V3.1']:
    r = earthaccess.search_data(short_name=sn, temporal=('2023-01-01', '2023-01-02'))
    print(('OK ' if r else 'ERR'), sn, len(r), 'granules')

In [ ]:
# ── Dry run: one month of every input into a scratch store, then look at it ──
from oceanembed import ingest
C.START, C.END = '2023-01-01', '2023-01-31'
ingest.ingest_inputs('/content/dry', days_per_block=31)
import xarray as xr, matplotlib.pyplot as plt
d = xr.open_zarr('/content/dry/inputs.zarr').isel(time=14)
fig, ax = plt.subplots(2, 4, figsize=(20, 7))
for a, v in zip(ax.flat, C.INPUT_VARS): d[v].plot(ax=a, cmap='RdYlBu_r'); a.set_title(v)
ax.flat[-1].axis('off'); plt.tight_layout()
C.START, C.END = '2014-01-01', '2023-12-31'      # restore the full 10-year period

## Full 10-year ingest (long: run, close the tab, come back — it resumes)

In [ ]:
ingest.ingest_inputs(ROOT, days_per_block=31)

In [ ]:
# GLORYS target: streamed from Copernicus ARCO, vertical → 15 std depths, 1/12° → 0.25° area mean.
ingest.ingest_glorys(ROOT, days_per_block=8)

In [ ]:
# Independent comparator from YOUR Google Earth Engine account (not used in training)
ingest.ingest_hycom(ROOT)

In [ ]:
# ── Independent Argo profiles for the test year ────────────────────────────
from oceanembed import argo
prof = argo.fetch_profiles()
prof.to_parquet(f'{ROOT}/argo_2023.parquet'); print(len(prof), 'profiles')

### Optional — Stage-1 transfer learning data (paper §2.2.2)
Export **monthly gridded Argo** temperature for 2014–2023 over 45–105°E, 5–30°N from the INCOIS LAS
(the PS's named source) as NetCDF, upload to `MyDrive/OceanEmbed/argo_grid/`, set `VAR` to its variable name.

In [ ]:
import glob
files = sorted(glob.glob(f'{ROOT}/argo_grid/*.nc'))
VAR = 'TEMP'                      # check with xr.open_dataset(files[0])
if files: ingest.ingest_gridded_argo(ROOT, files, VAR)
else: print('no gridded Argo files yet: stage 1 will be skipped')

In [ ]:
# ── QA: coverage per variable (fraction of ocean pixel-days that are valid) ──
ds = xr.open_zarr(f'{ROOT}/inputs.zarr'); T = xr.open_zarr(f'{ROOT}/target.zarr').thetao
ocean = T.isel(time=0, depth=0).notnull()
for v in C.INPUT_VARS:
    frac = float(ds[v].where(ocean).notnull().sum() / (ocean.sum() * ds.sizes['time']))
    print(f'{v:4s} {frac:6.1%}')
print('target', float(T.isel(depth=0).where(ocean).notnull().sum() / (ocean.sum() * T.sizes['time'])))